In [7]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (SparkSession.builder
    .appName("chakhan-price-cluster")
    .master("spark://jupyter:7077")
    .getOrCreate())

print("master:", spark.sparkContext.master)
print("등록된 executor(driver 포함):", spark.sparkContext._jsc.sc().getExecutorMemoryStatus().keySet().toString())

spark

master: spark://jupyter:7077
등록된 executor(driver 포함): Set(5597bfd010a9:33913, 172.18.0.3:34109, 172.18.0.4:43237)


In [4]:
# 셀 — CSV 로드

In [8]:
PATH = "/home/jovyan/work/행정안전부_착한가격업소 현황_20260630.csv"

df = (spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("encoding", "UTF-8")     # 깨지면 UTF-8로 교체
    .option("multiLine", True)
    .option("escape", '"')
    .csv(PATH))

df.printSchema()
print("행 수:", df.count())
print(df.columns)

root
 |-- 시도: string (nullable = true)
 |-- 시군: string (nullable = true)
 |-- 업종: string (nullable = true)
 |-- 업소명: string (nullable = true)
 |-- 연락처: string (nullable = true)
 |-- 주소: string (nullable = true)
 |-- 메뉴1: string (nullable = true)
 |-- 가격1: string (nullable = true)
 |-- 메뉴2: string (nullable = true)
 |-- 가격2: string (nullable = true)
 |-- 메뉴3: string (nullable = true)
 |-- 가격3: string (nullable = true)
 |-- 메뉴4: string (nullable = true)
 |-- 가격4: string (nullable = true)

행 수: 12645
['시도', '시군', '업종', '업소명', '연락처', '주소', '메뉴1', '가격1', '메뉴2', '가격2', '메뉴3', '가격3', '메뉴4', '가격4']


In [9]:
# 셀 3 — 컬럼명 영문화 + 가격 타입 통일

In [10]:
rename_map = {
    "시도": "sido", "시군": "sigun", "업종": "category",
    "업소명": "name", "연락처": "tel", "주소": "address",
    "메뉴1": "menu1", "가격1": "price1",
    "메뉴2": "menu2", "가격2": "price2",
    "메뉴3": "menu3", "가격3": "price3",
    "메뉴4": "menu4", "가격4": "price4",
}
for old, new in rename_map.items():
    df = df.withColumnRenamed(old, new)

# ★ 가격 컬럼 타입 통일 (inferSchema가 컬럼마다 int/string 다르게 잡았을 수 있음)
for c in ["price1", "price2", "price3", "price4"]:
    df = df.withColumn(c, F.col(c).cast("string"))

df.limit(5).toPandas()

,sido,sigun,category,name,tel,address,menu1,price1,menu2,price2,menu3,price3,menu4,price4
0,서울특별시,종로구,양식,돈까스보라,02-741-3455,서울특별시 종로구 대학로5길 5 (연건동),수제 돈까스,7000,돈까스 정식,8000,None,None,None,None
1,서울특별시,종로구,한식,삼삼 뚝배기,02-765-4683,서울특별시 종로구 동숭길 51 (동숭동),된장뚝배기,7500,김치찌개,7500,None,None,None,None
2,서울특별시,종로구,기타요식업,약속커피숍,02-764-1100,서울특별시 종로구 종로 302 (창신동),맥심커피,2000,녹차,2000,None,None,None,None
3,서울특별시,종로구,미용업,영미용실,02-743-6052,서울특별시 종로구 지봉로 62 (숭인동),커트,8000,파마,20000,None,None,None,None
4,서울특별시,종로구,기타요식업,왕관커피숍,02-2266-8365,서울특별시 종로구 종로 222 (종로5가),커피,2000,국산차,2000,None,None,None,None


In [11]:
# 셀 4 — Wide → Long 변환 (핵심)

In [12]:
long_df = (df.select(
        "sido", "sigun", "category", "name", "address",
        F.expr("""
            stack(4,
                1, menu1, price1,
                2, menu2, price2,
                3, menu3, price3,
                4, menu4, price4
            ) as (menu_no, menu, price_raw)
        """)
    )
    .filter(F.col("menu").isNotNull() & (F.trim(F.col("menu")) != ""))
    # 1) 첫 번째 숫자 덩어리만 추출 (콤마 포함 허용)
    .withColumn("price_str",
        F.regexp_extract(F.col("price_raw"), r"(\d{1,3}(?:,\d{3})+|\d+)", 1))
    # 2) 콤마 제거 후 try_cast → 실패해도 NULL, 예외 안 남
    .withColumn("price",
        F.expr("try_cast(regexp_replace(price_str, ',', '') as bigint)"))
    .filter(F.col("price").isNotNull() & (F.col("price") > 0))
    # 3) 상식 밖 값 차단 (10원 미만 / 100만원 초과)
    .filter((F.col("price") >= 500) & (F.col("price") <= 1000000))
    .drop("price_str")
)

long_df.cache()
print("메뉴 단위 행 수:", long_df.count())
long_df.limit(10).toPandas()

메뉴 단위 행 수: 26762


,sido,sigun,category,name,address,menu_no,menu,price_raw,price
0,서울특별시,종로구,양식,돈까스보라,서울특별시 종로구 대학로5길 5 (연건동),1,수제 돈까스,7000,7000
1,서울특별시,종로구,양식,돈까스보라,서울특별시 종로구 대학로5길 5 (연건동),2,돈까스 정식,8000,8000
2,서울특별시,종로구,한식,삼삼 뚝배기,서울특별시 종로구 동숭길 51 (동숭동),1,된장뚝배기,7500,7500
3,서울특별시,종로구,한식,삼삼 뚝배기,서울특별시 종로구 동숭길 51 (동숭동),2,김치찌개,7500,7500
4,서울특별시,종로구,기타요식업,약속커피숍,서울특별시 종로구 종로 302 (창신동),1,맥심커피,2000,2000
5,서울특별시,종로구,기타요식업,약속커피숍,서울특별시 종로구 종로 302 (창신동),2,녹차,2000,2000
6,서울특별시,종로구,미용업,영미용실,서울특별시 종로구 지봉로 62 (숭인동),1,커트,8000,8000
7,서울특별시,종로구,미용업,영미용실,서울특별시 종로구 지봉로 62 (숭인동),2,파마,20000,20000
8,서울특별시,종로구,기타요식업,왕관커피숍,서울특별시 종로구 종로 222 (종로5가),1,커피,2000,2000
9,서울특별시,종로구,기타요식업,왕관커피숍,서울특별시 종로구 종로 222 (종로5가),2,국산차,2000,2000


In [13]:
# 셀 5 — 분산 처리 증거 1: 파티션별 처리 worker(hostname) 확인

In [14]:
import socket
import pandas as pd

def hostname_of_partition(idx, iterator):
    yield (idx, socket.gethostname())

# 파티션 수를 워커 수(2)보다 넉넉히 늘려서 여러 워커에 고르게 흩어지도록 함
long_df_repart = long_df.repartition(6)

partition_hosts = (long_df_repart.rdd
    .mapPartitionsWithIndex(hostname_of_partition)
    .collect())

host_df = pd.DataFrame(partition_hosts, columns=["partition_id", "executor_hostname"])
print(host_df)
print()
print("파티션 처리 worker 분포 (hostname이 2개 이상 나오면 실제로 두 worker가 나눠 처리한 것):")
print(host_df["executor_hostname"].value_counts())

   partition_id executor_hostname
0             0      0ccd9210badb
1             1      0ccd9210badb
2             2      0ccd9210badb
3             3      0ccd9210badb
4             4      0ccd9210badb
5             5      0ccd9210badb

파티션 처리 worker 분포 (hostname이 2개 이상 나오면 실제로 두 worker가 나눠 처리한 것):
executor_hostname
0ccd9210badb    6
Name: count, dtype: int64


In [15]:
# 셀 6 — 분산 처리 증거 2: 마스터가 인식하는 worker 목록 (Spark Master REST API)

In [16]:
import urllib.request, json

master_state = json.loads(urllib.request.urlopen("http://jupyter:8080/json/").read())

print("alive workers:", master_state["aliveworkers"])
for w in master_state["workers"]:
    print(f"  - id={w['id']}  host={w['host']}  state={w['state']}  cores={w['cores']}  memory={w['memory']}MB")

print()
print("running/completed apps:", len(master_state["activeapps"]), "/", len(master_state["completedapps"]))

alive workers: 2
  - id=worker-20260728054832-172.18.0.4-35295  host=172.18.0.4  state=ALIVE  cores=1  memory=1500MB
  - id=worker-20260728054832-172.18.0.3-43565  host=172.18.0.3  state=ALIVE  cores=1  memory=1500MB

running/completed apps: 1 / 1


In [17]:
# 결론
# - 셀 5: partition_id별로 서로 다른 hostname(=spark-worker-1, spark-worker-2 컨테이너)이 찍히면
#   실제로 두 worker가 데이터를 나눠서 처리했다는 직접적 증거.
# - 셀 6: 마스터 REST API(/json/)로 조회한 aliveworkers 수와 각 worker의 host/cores/memory가
#   docker-compose.yml에 정의한 spark-worker-1, spark-worker-2와 일치하는지 대조.
# - 01/02(local[*])는 이 두 셀을 실행해도 hostname이 jupyter 컨테이너 하나로만 찍히고,
#   마스터 자체가 없으므로 :8080/json/ 호출도 실패한다는 점이 01,02와의 핵심 차이.